# 11장 — 🎯 캡스톤: 파인튜닝 설정 완독 🎓

졸업 시험: 12조합 그리드 서치와 설정 파일 완독을 코드로 해냅니다.

**이 노트북의 목표**
1. 미니 조련 튜너: 다이얼 3종(보폭·일정표·목줄) 12조합 그리드 서치, 합격 조합 찾기
2. 합격 조합이 하나뿐인 이유: 각 다이얼이 서로 다른 실패를 막는다
3. 설정 파일 12줄 완독: 모든 줄이 몇 장의 수학인지 확인 (모르는 줄 0)

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch11')

## 1. 졸업 시험 ①: 미니 조련 튜너 (그리드 서치)

7~8장의 무대(시드 21/22) + 노이즈 낀 기울기. 다이얼: lr {0.0003, 0.05, 2.0} ×
일정표 {고정, 워밍업+코사인} × 목줄 λ {0, 0.03}.
**미션: 검증 손실(마지막 100스텝 평균) ≤ 0.10**. 이 무대의 합격선입니다
(합격선은 문제마다 다릅니다. 웹 무대는 0.15였어요).

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')   # 발산 조합의 overflow 경고는 예상된 것

def make_data(n, seed):
    rng = np.random.default_rng(seed)
    x = np.clip(np.linspace(-1, 1, n) + rng.uniform(-0.04, 0.04, n), -1, 1)
    return x, np.sin(2.5 * x) + 0.25 * rng.standard_normal(n)

def cheb(x):
    T = [np.ones_like(x), x]
    for k in range(2, 10): T.append(2 * x * T[-1] - T[-2])
    return np.stack(T, axis=1)

xt, yt = make_data(10, 21); xv, yv = make_data(30, 22)
Ft, Fv = cheb(xt), cheb(xv)

def train(lr0, cosine, lam, noise_seed=6, sig=0.7, steps=3000):
    rng = np.random.default_rng(noise_seed)   # 노이즈 시드 공통 — 공정 비교
    w = np.zeros(10); vas = []
    for t in range(steps):
        lr = lr0
        if cosine:
            lr = lr0 * min(1, (t + 1) / 100) * 0.5 * (1 + np.cos(np.pi * t / steps))
        g = 2 * Ft.T @ (Ft @ w - yt) / 10
        w = w - lr * (g + 2 * lam * w + sig * rng.standard_normal(10))
        if not np.isfinite(w).all(): return None, None
        vas.append(np.mean((Fv @ w - yv) ** 2))
    return float(np.mean(vas[-100:])), vas

MISSION = 0.10
results = {}
print(f'{"lr":>7} {"일정표":<9} {"목줄":>5}   결과')
for lr in [0.0003, 0.05, 2.0]:
    for cos in [False, True]:
        for lam in [0, 0.03]:
            tail, vas = train(lr, cos, lam)
            results[(lr, cos, lam)] = (tail, vas)
            tag = '💥 발산' if tail is None else f'{tail:.3f} ' + ('✅ 합격' if tail <= MISSION else '❌')
            print(f'{lr:>7} {"코사인" if cos else "고정":<9} {lam:>5}   {tag}')

passes = [k for k, (t, _) in results.items() if t is not None and t <= MISSION]
diverged = [k for k, (t, _) in results.items() if t is None]

assert passes == [(0.05, True, 0.03)], '합격 조합은 단 하나 — 실전 기본값(적당 lr + 워밍업·코사인 + 목줄)'
assert len(diverged) == 4 and all(k[0] == 2.0 for k in diverged), '거인 걸음은 일정표·목줄로 못 구한다 (2장 경계)'
assert all(results[(0.0003, c, l)][0] > MISSION for c in [False, True] for l in [0, 0.03]), '잔걸음은 3000스텝에도 미도달 (소심이)'
print('\n✔ 확인: 12조합 중 합격은 단 하나. 각 다이얼이 서로 다른 실패(발산·미도달·서성임·벌어짐)를 하나씩 막는다')

## 2. 성적표 해부: 실패마다 진단이 붙는다 (9장 기술)

In [ ]:
picks = [
    ((0.05, False, 0),    'tab:red',    '고정+목줄X: 서성임+벌어짐'),
    ((0.05, False, 0.03), 'tab:orange', '고정+목줄: 서성임(감쇠 필요)'),
    ((0.05, True, 0),     'tab:purple', '코사인+목줄X: 얼기 전 암기'),
    ((0.05, True, 0.03),  'tab:green',  '코사인+목줄: ✅ 합격'),
]
plt.figure(figsize=(8, 3.4))
for key, c, label in picks:
    plt.plot(results[key][1], color=c, label=label, lw=1.2)
plt.axhline(MISSION, color='gray', ls=':', label=f'미션 {MISSION}')
plt.ylim(0, 1.0); plt.xlabel('스텝'); plt.ylabel('검증 손실')
plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.title('같은 보폭(0.05), 다른 운명: 일정표와 목줄이 가른다')
plt.tight_layout(); plt.show()

t_ff, t_fd = results[(0.05, False, 0)][0], results[(0.05, False, 0.03)][0]
t_cf, t_cd = results[(0.05, True, 0)][0], results[(0.05, True, 0.03)][0]
print(f'고정+목줄X {t_ff:.3f} → 목줄만 {t_fd:.3f} → 코사인만 {t_cf:.3f} → 둘 다 {t_cd:.3f}')

assert t_cd < t_fd and t_cd < t_cf, '합격에는 둘 다 필요 — 목줄은 벌어짐(7장)을, 감쇠는 서성임(6장)을 막는다'
assert t_fd > MISSION and t_cf > MISSION, '하나만 채우면 남은 병으로 탈락'
print('\n✔ 확인: 다이얼마다 담당하는 병이 다르다. 기본값 조합(adamw·cosine·warmup·decay)이 기본값인 이유')

## 3. 졸업 시험 ②: 설정 파일 완독

6부 3장에서 스쳐 지나간 그 설정 파일, 이제 모든 줄에 담당 수학을 답할 수 있어야 합니다.

In [ ]:
config = {
    'model': 'llama-3-8b',
    'dataset': 'company_qa_1200',
    'num_epochs': 3,
    'batch_size': 16,
    'learning_rate': 2e-5,
    'lr_scheduler': 'cosine',
    'warmup_steps': 100,
    'optimizer': 'adamw',
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    'eval_steps': 50,
    'early_stopping_patience': 3,
}
chapter_of = {
    'model': '6부 (+ 1·10장: 80억 파라미터 = 지형의 차원)',
    'dataset': '6부 3장 (SFT 데이터, 소량이니 과적합 경계: 7장)',
    'num_epochs': '3장 에포크 · 7장 과적합의 첫 다이얼',
    'batch_size': '3장 (나침반 흔들림 σ/√B)',
    'learning_rate': '2장 (보폭: 수십억 차원의 예민한 경계 2/c)',
    'lr_scheduler': '6장 (여행 일정표: 코사인 감쇠)',
    'warmup_steps': '6장 워밍업 (+ 5장 Adam 자 예열)',
    'optimizer': '5장 Adam(4장 모멘텀 + RMS) + 8장 W(분리된 감쇠)',
    'weight_decay': '8장 (원점 목줄: 6부 6장 KL 목줄의 형제)',
    'max_grad_norm': '9장 (그래디언트 클리핑: 방향 유지·크기 절단)',
    'eval_steps': '7장 검증 + 9장 훈련 일지 (없으면 암기가 안 보임)',
    'early_stopping_patience': '8장 조기 종료 (7장 벌어짐의 자동 처방)',
}

print('설정 파일 완독, 줄마다 담당 수학:')
for k, v in config.items():
    print(f'  {k:>24}: {str(v):<16} ← {chapter_of[k]}')

unread = [k for k in config if k not in chapter_of]
assert unread == [], f'모르는 줄이 남아 있다: {unread}'

# 규모 계산 — 이제 한 줄 (StepQuiz 값)
steps_per_epoch = 1200 // config['batch_size']
total_steps = steps_per_epoch * config['num_epochs']
print(f'\n총 스텝 수 = 1200/{config["batch_size"]} × {config["num_epochs"]} = {total_steps}')
assert steps_per_epoch == 75 and total_steps == 225
print('\n🎓 완독! 설정 파일에 모르는 줄이 없습니다. 8부 졸업!')

## 4. 졸업 정리: 열한 장이 12줄로

- **미션의 교훈**: 합격 조합은 단 하나(적당 lr + 워밍업·코사인 + 목줄). 각 다이얼이 서로
  다른 실패를 막는다: lr → 발산·미도달(2장), 감쇠 → 서성임(6장), 목줄 → 벌어짐(7·8장).
  거인 걸음은 무엇으로도 못 구한다 → **튜닝은 lr 자릿수부터**, 그다음 한 번에 하나(9장).
- **완독의 의미**: 설정 파일의 모든 줄은 시리즈 어딘가의 수학, 곧 보폭(2장)·나침반(3장)·
  관성과 축별 자(4·5장)·일정표(6장)·봉인된 시험(7장)·목줄(8장)·안전핀과 일지(9장)·
  지형(1·10장).
- **6부와 8부**: 무엇을 배우게 할까(조련) + 어떻게 잘 배우게 할까(다이얼), 조련사의 두 손.
- 다음 여정의 재료(이름만): 분산 훈련, 혼합 정밀도(fp16/bf16), 스케일링 법칙.

**🎉 8부 완결**. 수포자를 위한 최적화, 워밍업 + 11장의 여행이 끝났습니다.
훈련 일지 앞에서 곡선이 말을 걸고, 설정 파일에 모르는 줄이 없기를.